In [1]:
import warnings
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Literal

from dotenv import load_dotenv
from google.genai import Client as genaiClient
from google.genai import types
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_core._api import LangChainDeprecationWarning
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.retrievers import BaseRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)

In [2]:
_ = load_dotenv()
llm = init_chat_model("deepseek-chat")

In [3]:
class ResearchSchema(BaseModel):
    answer: str = Field(description="The answer to the question")
    confidence: Literal["high", "medium", "low"] = Field(description="How confident are you in the answer")
    sources: list[str] = Field(description="A list of sources the answer came from")
    key_quotes: list[str] = Field(description="Relevant quotes from the source", default=[])
    follow_up: list[str] = Field(description="Suggested follow up questions", default=[])

In [4]:

class GeminiEmbeddings(Embeddings):
    def __init__(self, model_name: str = "gemini-embedding-2", dimensions: int = 3072):
        self.model_name = model_name
        self.dimensions = dimensions
        self.client = genaiClient()

    def _embed(self, text: str) -> list[float]:
        result = self.client.models.embed_content(
            model=self.model_name,
            contents=text,
            config=types.EmbedContentConfig(output_dimensionality=self.dimensions),
        )
        return result.embeddings[0].values

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(t) for t in texts]


embeddings = GeminiEmbeddings()

In [13]:
@dataclass
class ResearchAssistant:
    persist_directory: str = "./research_db"
    chunk_size: int = 1000
    chunk_overlap: int = 100

    _embeddings: Any = field(default=None, init=False, repr=False)
    _vectorstore: Any = field(default=None, init=False, repr=False)
    _retriever: Any = field(default=None, init=False, repr=False)

    llm: Any = field(default_factory=lambda: init_chat_model("deepseek-chat", temperature=0))

    def __post_init__(self):
        self.splitter = RecursiveCharacterTextSplitter(chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap)
        self.session_store: dict[str, InMemoryChatMessageHistory] = {}

    @property
    def embeddings(self) -> GeminiEmbeddings:
        if self._embeddings is None:
            self._embeddings = GeminiEmbeddings()
        return self._embeddings

    @property
    def load_vectorstore(self) -> Chroma:
        if self._vectorstore is None:
            self._vectorstore = Chroma(
                persist_directory=self.persist_directory,
                embedding_function=self.embeddings,
                collection_name="research_docs"
            )
        return self._vectorstore

    def get_retriever(self, use_advanced: bool = False) -> BaseRetriever:
        if self._retriever is None:
            self._retriever = self.load_vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})
        if use_advanced:
            return MultiQueryRetriever.from_llm(retriever=self._retriever, llm=self.llm)
        return self._retriever

    def add_documents(self, documents: list[Document], source_name: str | None = None) -> int:
        if source_name:
            for doc in documents:
                doc.metadata["source"] = source_name
        chunks = self.splitter.split_documents(documents)
        for chunk in chunks:
            chunk.metadata["indexed_at"] = datetime.now().isoformat()
        self.load_vectorstore.add_documents(chunks)
        print(f"Added {len(chunks)} chunks from {len(documents)} documents")
        return len(chunks)

    def add_text(self, text: str, source: str, metadata: dict | None = None) -> int:
        doc = Document(page_content=text, metadata={"source": source, **(metadata or {})})
        return self.add_documents([doc])

    def add_texts(self, texts: list[str], source: str) -> int:
        docs = [Document(page_content=t, metadata={"source": source}) for t in texts]
        return self.add_documents(docs)

    def get_document_count(self) -> int:
        return self.load_vectorstore._collection.count()

    def list_sources(self) -> list[str]:
        results = self.load_vectorstore._collection.get()
        metadatas = results.get("metadatas") or []
        return sorted({m["source"] for m in metadatas if m and "source" in m})

    def ask(self, question: str, session_id: str = "default", use_advanced: bool = True) -> ResearchSchema:
        docs = self.get_retriever(use_advanced).invoke(question)
        context = self._format_docs_for_context(docs)
        history = self._get_session_history(session_id)
        chain = self._build_prompt() | self.llm.with_structured_output(ResearchSchema)
        result: ResearchSchema = chain.invoke(
            {"context": context, "question": question, "history": history.messages[-10:]})
        history.add_message(HumanMessage(content=question))
        history.add_message(AIMessage(content=result.model_dump_json()))
        return result

    def print_response(self, result: ResearchSchema, question: str | None = None) -> None:
        width = 70
        print("=" * width)
        if question:
            print(f"Q: {question}")
        print(f"[{result.confidence.upper()}] {result.answer}")
        if result.key_quotes:
            print(f"Quotes:{' | '.join(result.key_quotes)}")
        if result.sources:
            print(f"Sources: {' | '.join(result.sources)}")
        if result.follow_up:
            print(f"Follow: {' | '.join(result.follow_up)}")
        print("=" * width)

    def get_session_messages(self, session_id: str) -> list[BaseMessage]:
        return self._get_session_history(session_id).messages

    def clear_session(self, session_id: str) -> None:
        if session_id in self.session_store:
            self.session_store.pop(session_id)

    def _get_session_history(self, session_id: str) -> InMemoryChatMessageHistory:
        if session_id not in self.session_store:
            self.session_store[session_id] = InMemoryChatMessageHistory()
        return self.session_store[session_id]

    def _format_docs_for_context(self, docs: list[Document]) -> str:
        if not docs:
            return "No relevant documents found."
        return "\n\n---\n\n".join(
            f"[Source {i}: {d.metadata.get('source', 'Unknown')}]\n{d.page_content}"
            for i, d in enumerate(docs, 1)
        )

    def _build_prompt(self) -> ChatPromptTemplate:
        return ChatPromptTemplate.from_messages([
            ("system", "You are a research assistant. Use the following retrieved context to answer "
                       "the user's question. If the context doesn't contain enough information, say so.\n\n"
                       "Context:\n{context}"),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{question}"),
        ])

In [7]:
assistant = ResearchAssistant(persist_directory="./research_db")

with open("docs/stats.txt", "r", encoding="utf-8") as f:
    file_content = f.read()

print("Ingesting Foundations of Statistics text...")
assistant.add_text(
    text=file_content,
    source="foundations_of_stats.txt",
    metadata={"category": "education"}
)


Ingesting Foundations of Statistics text...
Added 32 chunks from 1 documents


32

In [14]:
test_queries = {
    "Single-Hop": "What is the exact mathematical rule used to define outliers?",
    "Multi-Hop": "What are the non-parametric alternatives to the two-sample t-test and ANOVA?",
    "Edge-Case": "What common misconception about 95% Confidence Intervals does the text explicitly warn against?"
}

print("Structured RAG Evaluation Scenarios")
for q_type, query in test_queries.items():
    print(f"\n[{q_type}]")
    result = assistant.ask(query)
    assistant.print_response(result, question=query)

Structured RAG Evaluation Scenarios

[Single-Hop]
Q: What is the exact mathematical rule used to define outliers?
[HIGH] The exact mathematical rule used to define outliers is based on the interquartile range (IQR). A value is considered an outlier if it falls below Q1 - 1.5*IQR or above Q3 + 1.5*IQR, where Q1 is the 25th percentile, Q3 is the 75th percentile, and IQR = Q3 - Q1.
Quotes:The IQR is used to define outliers: a value is considered an outlier if it falls below Q1 - 1.5*IQR or above Q3 + 1.5*IQR.
Sources: Source 3: foundations_of_stats.txt

[Multi-Hop]
Q: What are the non-parametric alternatives to the two-sample t-test and ANOVA?
[HIGH] The non-parametric alternative to the two-sample t-test is the Wilcoxon rank-sum test (also known as the Mann-Whitney U test). The non-parametric alternative to one-way ANOVA is the Kruskal-Wallis test.
Quotes:The **Wilcoxon rank-sum test** (Mann-Whitney U test) is a non-parametric alternative to the two-sample t-test. | The **Kruskal-Wallis 